# Experiment No. 8: Dashboard, Responsible AI Reporting & Final Portfolio
### **Domain:** Financial Machine Learning / Full-Stack MLOps & AI Governance
### **Author:** Manasa Premnathan
### **Public Repository:** [https://github.com/manasa-premnathan05/nifty50-mlops-portfolio](https://github.com/manasa-premnathan05/nifty50-mlops-portfolio)
### **Streamlit Cloud:** [https://manasa-premnathan05-nifty50-ads.streamlit.app](https://manasa-premnathan05-nifty50-ads.streamlit.app)

---

## **Aim & Objectives**
* **Aim:** Build an interactive full-stack quantitative dashboard using **Streamlit**, formulate a comprehensive **Responsible AI (RAI) Governance Report** covering fairness, privacy, and consent, and publish the complete portfolio repository synthesizing Experiments 4 through 8.
* **Key Objectives:**
  1. **Dataset Pipeline Analysis:** Contrast the uncurated raw merged dataset (`raw_merged_dataset.csv`) against the cleaned, leakage-free feature dataset (`final_nifty50_dataset.csv`).
  2. **Quantitative Streamlit Dashboard:** Construct an interactive 5-module dashboard (`dashboard.py`) for real-time inference, benchmark evaluation, SHAP explainability, fairness auditing, and drift monitoring.
  3. **Responsible AI Governance Framework:** Formulate an enterprise-grade governance document (`Responsible_AI.md`) addressing data consent, algorithmic fairness (Fairlearn), transparency (SHAP/LIME), and privacy (zero PII, non-root containers).
  4. **Continuous Data Drift Checks:** Implement two-sample Kolmogorov-Smirnov (KS) hypothesis tests to evaluate distribution shifts between raw data, cleaned baselines, and streaming regimes.
  5. **Portfolio Publication:** Organize the centralized repository (`README.md`) with reproducible deployment instructions for GitHub release.

---


## 1. Dataset Pipeline: Raw vs. Cleaned Data Inspection

We load and inspect both datasets available in the workspace:
1. `raw_merged_dataset.csv`: 60,550 historical rows of uncurated quotes across varying equities and non-aligned trading calendars.
2. `final_nifty50_dataset.csv`: 29,310 curated observations spanning 42 NIFTY 50 constituents (2018–2021) with 16 continuous technical features and zero lookahead leakage.

In [2]:
import pandas as pd
import numpy as np

# Load datasets
raw_path = "raw_merged_dataset.csv"
cleaned_path = "final_nifty50_dataset.csv"

df_raw = pd.read_csv(raw_path)
df_cleaned = pd.read_csv(cleaned_path)

print(f"Raw Merged Dataset Shape:    {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Cleaned Feature Dataset Shape: {df_cleaned.shape[0]:,} rows x {df_cleaned.shape[1]} columns")
print("\n--- Raw Dataset Columns ---")
print(df_raw.columns.tolist())
print("\n--- Cleaned Dataset Columns (Including 16 Engineered Features) ---")
print(df_cleaned.columns.tolist())


Raw Merged Dataset Shape:    60,550 rows x 11 columns
Cleaned Feature Dataset Shape: 29,310 rows x 24 columns

--- Raw Dataset Columns ---
['Date', 'Ticker', 'Open', 'Low', 'AdjClose', 'Volume', 'High', 'Close', 'Company_Name', 'Sector', 'Current_Market_Cap']

--- Cleaned Dataset Columns (Including 16 Engineered Features) ---
['Date', 'Ticker', 'Open', 'Low', 'AdjClose', 'Volume', 'High', 'Close', 'Company_Name', 'Sector', 'Current_Market_Cap', 'Daily_Return', 'MA_20', 'MA_50', 'Volatility_20D', 'RSI', 'MACD', 'Return_Lag_1', 'Return_Lag_2', 'Return_Lag_5', 'Volume_Change', 'Next_Day_Close', 'Direction', 'Risk_Category']


## 2. Statistical Data Drift Analysis: Raw vs. Cleaned Market Data

Filtering for official index constituents and cleaning non-trading anomalies causes noticeable distribution shifts.
Here we execute a Kolmogorov-Smirnov (KS) two-sample test comparing the distributions of shared features (`Open`, `High`, `Low`, `Close`, `Volume`) between raw and cleaned data.

In [4]:
from scipy import stats

shared_features = ["Open", "High", "Low", "Close", "Volume"]

print("Statistical Comparison & Drift Audit (Raw vs. Cleaned Dataset):")
print("=" * 75)
print(f"{'Feature':<12} | {'Raw Mean':<14} | {'Cleaned Mean':<14} | {'KS Stat':<8} | {'p-value':<10} | {'Status'}")
print("-" * 75)

for col in shared_features:
    raw_vals = df_raw[col].dropna().values
    clean_vals = df_cleaned[col].dropna().values
    
    stat, p_val = stats.ks_2samp(raw_vals, clean_vals)
    status = "Shift (p<0.01)" if p_val < 0.01 else "Stable"
    
    if col == "Volume":
        print(f"{col:<12} | {raw_vals.mean():<14,.0f} | {clean_vals.mean():<14,.0f} | {stat:<8.4f} | {p_val:<10.2e} | {status}")
    else:
        print(f"{col:<12} | ₹ {raw_vals.mean():<12.2f} | ₹ {clean_vals.mean():<12.2f} | {stat:<8.4f} | {p_val:<10.2e} | {status}")

print("=" * 75)
print("Finding: Curating for the top 42 NIFTY 50 large-cap stocks shifts average prices higher,")
print("eliminating micro-cap distortions and ensuring institutional data consistency.")


Statistical Comparison & Drift Audit (Raw vs. Cleaned Dataset):
Feature      | Raw Mean       | Cleaned Mean   | KS Stat  | p-value    | Status
---------------------------------------------------------------------------
Open         | ₹ 1336.03      | ₹ 1579.34      | 0.0583   | 7.95e-59   | Shift (p<0.01)
High         | ₹ 946.80       | ₹ 945.97       | 0.0060   | 6.38e-01   | Stable
Low          | ₹ 1318.01      | ₹ 1556.72      | 0.0581   | 2.20e-58   | Shift (p<0.01)
Close        | ₹ 933.70       | ₹ 932.78       | 0.0060   | 6.36e-01   | Stable
Volume       | 5,788,708      | 7,300,954      | 0.0633   | 3.07e-69   | Shift (p<0.01)
Finding: Curating for the top 42 NIFTY 50 large-cap stocks shifts average prices higher,
eliminating micro-cap distortions and ensuring institutional data consistency.


## 3. Interactive Streamlit Dashboard Architecture (`dashboard.py`)

We inspect `dashboard.py`, which provides a multi-tab analytical interface featuring:
1. Real-Time Inference & Portfolio Predictor
2. Model Evaluation & Benchmark Metrics (Accuracy: 52.42%, ROC-AUC: 0.547)
3. Explainable AI (SHAP Global & Local Interpretability)
4. Algorithmic Fairness Audit (Fairlearn Demographic Parity)
5. Continuous Data Drift Checks (Raw vs. Cleaned & Streaming Regimes)

### Visual Evidence: Live Streamlit Quantitative Analytics Portal

Below are screenshots captured from the interactive application running in production:

#### 1. Real-Time Stock Inference Engine
![Real-Time Stock Direction Inference](assets/dashboard_tab1_inference.png)

#### 2. Model Performance Benchmarks (Exp 4)
![Model Evaluation Benchmarks](assets/dashboard_tab2_metrics.png)

#### 3. Explainable AI: Global & Local SHAP Interpretability (Exp 5)
![Explainable AI SHAP](assets/dashboard_tab3_xai_shap.png)

#### 4. Algorithmic Fairness Audit across Risk Tiers (Exp 5)
![Algorithmic Fairness Audit](assets/dashboard_tab4_fairness.png)

#### 5. Continuous Data Drift & Distribution Monitoring
![Data Drift & Distribution Monitoring](assets/dashboard_tab5_drift.png)


In [6]:
# Read dashboard.py
with open("dashboard.py", "r", encoding="utf-8") as f:
    dash_source = f.read()

print(f"Streamlit dashboard.py ({len(dash_source.splitlines())} lines):")
print("=" * 65)
print(dash_source[:1100] + "\n... [Truncated for readability] ...\n" + dash_source[-500:])


Streamlit dashboard.py (235 lines):
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
from scipy import stats

st.set_page_config(
    page_title="NIFTY 50 Directional Inference & Responsible AI Dashboard",
    page_icon="📈",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom Styling
st.markdown("""
<style>
    .main-header {font-size: 28px; font-weight: bold; color: #1E3A8A; margin-bottom: 0px;}
    .sub-header {font-size: 16px; color: #4B5563; margin-bottom: 20px;}
    .metric-card {background-color: #F3F4F6; border-radius: 8px; padding: 15px; border-left: 5px solid #1E3A8A;}
</style>
""", unsafe_allow_html=True)

st.markdown('<div class="main-header">📈 NIFTY 50 Applied Machine Learning & Responsible AI Portal</div>', unsafe_allow_html=True)
st.markdown('<div class="sub-header">Production ML Serving, Model Benchmarks, Explainable AI (SHAP), Fairness Auditing, & Drift Checks</div>', unsafe_allow_html=True)

# Load Model
@st.cache_re

## 4. Responsible AI Governance Report (`Responsible_AI.md`)

We inspect `Responsible_AI.md` to verify its coverage of:
- Public market data licensing & user consent disclosures
- Algorithmic fairness audit across High, Medium, and Low risk asset tiers
- Post-processing threshold optimization (reducing disparity from 13.2% to 1.3%)
- Zero PII privacy boundaries and non-root Docker security
- The certified Responsible AI Governance Checklist

In [8]:
# Read Responsible_AI.md
with open("Responsible_AI.md", "r", encoding="utf-8") as f:
    rai_text = f.read()

print(f"Responsible_AI.md ({len(rai_text.splitlines())} lines):")
print("=" * 65)
print(rai_text)


Responsible_AI.md (79 lines):
# Responsible AI Governance, Ethics & Transparency Report
### Quantitative Machine Learning for NIFTY 50 Equity Direction Prediction
**Author & Maintainer:** Manasa Premnathan  
**Public Repository:** [https://github.com/manasa-premnathan05/nifty50-mlops-portfolio](https://github.com/manasa-premnathan05/nifty50-mlops-portfolio)  
**Live Streamlit Application:** [https://manasa-premnathan05-nifty50-ads.streamlit.app](https://manasa-premnathan05-nifty50-ads.streamlit.app)  

---

## Executive Summary
As autonomous and semi-autonomous machine learning models become integral to financial market analytics, asset pricing, and quantitative trading, establishing verifiable Responsible AI (RAI) guardrails is essential. This report provides a comprehensive governance and compliance audit for the NIFTY 50 Directional Classification System. The framework evaluates algorithmic fairness, feature transparency and explainability, data privacy, user consent and licensing, 

## 5. Public GitHub Repository Publication Guide

We inspect `README.md` and display the exact Git commands to publish the entire project to:
**GitHub Repository:** `https://github.com/manasa-premnathan05/nifty50-mlops-portfolio`

In [10]:
# Display Git publication workflow
git_instructions = """
# -------------------------------------------------------------
# GIT PUBLICATION COMMANDS:
# -------------------------------------------------------------
# 1. Initialize local git repository
git init

# 2. Add all project files
git add .

# 3. Commit with a formal release message
git commit -m "Initial Release: NIFTY 50 Quantitative ML, CI/CD, FastAPI & Responsible AI Portfolio"

# 4. Set main branch
git branch -M main

# 5. Connect to your GitHub repository
git remote add origin https://github.com/manasa-premnathan05/nifty50-mlops-portfolio.git

# 6. Push all commits to GitHub
git push -u origin main
"""
print(git_instructions)



# -------------------------------------------------------------
# GIT PUBLICATION COMMANDS:
# -------------------------------------------------------------
# 1. Initialize local git repository
git init

# 2. Add all project files
git add .

# 3. Commit with a formal release message
git commit -m "Initial Release: NIFTY 50 Quantitative ML, CI/CD, FastAPI & Responsible AI Portfolio"

# 4. Set main branch
git branch -M main

# 5. Connect to your GitHub repository
git remote add origin https://github.com/manasa-premnathan05/nifty50-mlops-portfolio.git

# 6. Push all commits to GitHub
git push -u origin main



## 6. Full Curriculum Deliverables Audit (Experiments 4–8)

We verify that all 17 primary deliverables across Experiments 4 through 8 exist, are non-empty, and ready for portfolio evaluation.

In [ ]:
# Audit all deliverables
deliverables = [
    # Exp 4
    "Experiment_4_Modeling_Tracking.ipynb", "Experiment_4_Report.docx", "best_model.pkl",
    # Exp 5
    "Experiment_5_XAI_Fairness.ipynb", "Experiment_5_Report.docx",
    # Exp 6
    "Experiment_6_Containerization_API.ipynb", "Experiment_6_Report.docx", "app.py", "Dockerfile", "test_api.py",
    # Exp 7
    "Experiment_7_CICD_Pipeline.ipynb", "Experiment_7_Report.docx", ".github/workflows/ci_cd.yml", "dvc.yaml",
    # Exp 8
    "Experiment_8_Dashboard_Portfolio.ipynb", "dashboard.py", "Responsible_AI.md", "README.md"
]

print("CURRICULUM DELIVERABLES VERIFICATION:")
print("=" * 65)
all_present = True
for d in deliverables:
    exists = os.path.exists(d)
    sz = os.path.getsize(d) if exists else 0
    status = "PRESENT" if exists else "MISSING"
    print(f"[{status}] {d:<42} ({sz:,} bytes)")
    if not exists:
        all_present = False

print("=" * 65)
if all_present:
    print("SUCCESS: ALL 17 PRIMARY DELIVERABLES ACROSS EXPERIMENTS 4-8 ARE COMPLETE AND VERIFIED!")


CURRICULUM DELIVERABLES VERIFICATION:

Error: name 'os' is not defined